In [9]:
# ============================================================
# R1-25 : Dyadic dependence in the cross-community chi-square
# Input : network_edges.csv (13,994 unique dyads; source,target,weight,
#         first_interaction,last_interaction)
#         network_nodes_with_community3.csv (username;community;...)
# Output: unique-dyad chi-square + Fisher + dyad-level permutation
# ============================================================
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact

# ---- load ----
e = pd.read_csv('network_edges.csv', encoding='utf-8-sig', low_memory=False)
n = pd.read_csv('network_nodes_with_community3.csv', sep=';', encoding='utf-8-sig', low_memory=False)

# ---- normalize usernames identically on both sides, map to community ----
def norm(s):
    if pd.isna(s): return None
    return str(s).strip().lstrip('@').lower()

n['u'] = n['username'].map(norm)
comm = dict(zip(n['u'], n['community']))

e['sc'] = e['source'].map(norm).map(comm)
e['tc'] = e['target'].map(norm).map(comm)

# sanity: every edge should map (expect 13994 / 13994)
print(f"edges mapped to communities: {e[['sc','tc']].notna().all(axis=1).sum()} of {len(e)}")
e = e.dropna(subset=['sc','tc']).copy()

# ---- cross vs within (as strings so the crosstab column can never collapse) ----
e['cross'] = np.where(e['sc'] != e['tc'], 'cross', 'within')

# ---- period activity from edge timestamps ----
# pre  window : Aug 1-26   -> edge active in pre  if first_interaction <= Aug 26
# post window : Aug 28-Sep29 -> edge active in post if last_interaction  >= Aug 28
# Aug 27 excluded. A dyad active in both windows is counted once in each (correct
# for a pre-vs-post snapshot comparison; this is what removes repeated-interaction inflation).
e['fi'] = pd.to_datetime(e['first_interaction'])
e['li'] = pd.to_datetime(e['last_interaction'])
cut_pre  = pd.Timestamp('2025-08-26 23:59:59')
cut_post = pd.Timestamp('2025-08-28 00:00:00')
e['in_pre']  = e['fi'] <= cut_pre
e['in_post'] = e['li'] >= cut_post

rows = []
for _, r in e.iterrows():
    if r['in_pre']:  rows.append(('pre',  r['cross']))
    if r['in_post']: rows.append(('post', r['cross']))
dyad = pd.DataFrame(rows, columns=['period', 'cross'])

# ---- helper: build 2x2, run chi-square + Fisher + Cramer's V ----
def analyze(df, label):
    t = (pd.crosstab(df['period'], df['cross'])
           .reindex(index=['pre','post'], columns=['within','cross'])
           .fillna(0).astype(int))
    chi2, p, dof, _ = chi2_contingency(t.values, correction=False)
    N = t.values.sum()
    V = np.sqrt(chi2 / N)              # df=1 -> Cramer's V = sqrt(chi2/N)
    _, p_fisher = fisher_exact(t.values)
    print(f"\n== {label} ==")
    print(t)
    print(f"chi2={chi2:.2f}, p={p:.3g}, dof={dof}, Cramer's V={V:.3f}, Fisher p={p_fisher:.3g}")
    for per in ['pre','post']:
        tot = t.loc[per].sum()
        print(f"  {per}: cross%={100*t.loc[per,'cross']/tot:.1f}  (n={tot})")
    return t

# ---- MAIN: unique-dyad test ----
analyze(dyad, "UNIQUE DYAD (R1-25 main answer)")

# ---- ROBUSTNESS: dyad-level permutation test ----
rng = np.random.default_rng(42)
dyad['isc'] = (dyad['cross'] == 'cross').astype(float)
obs = dyad.loc[dyad.period=='post','isc'].mean() - dyad.loc[dyad.period=='pre','isc'].mean()
vals = dyad['isc'].values
n_post = int((dyad['period']=='post').sum())
B = 10000
cnt = 0
for _ in range(B):
    perm = rng.permutation(vals)
    if abs(perm[:n_post].mean() - perm[n_post:].mean()) >= abs(obs):
        cnt += 1
p_perm = (cnt + 1) / (B + 1)
print(f"\n== PERMUTATION (dyad-level, seed 42, B={B}) ==")
print(f"observed post-pre cross-prop diff = {obs:.4f};  p_perm = {p_perm:.4g}")

edges mapped to communities: 13994 of 13994

== UNIQUE DYAD (R1-25 main answer) ==
cross   within  cross
period               
pre       5706    965
post      5954   1248
chi2=21.17, p=4.2e-06, dof=1, Cramer's V=0.039, Fisher p=4.3e-06
  pre: cross%=14.5  (n=6671)
  post: cross%=17.3  (n=7202)

== PERMUTATION (dyad-level, seed 42, B=10000) ==
observed post-pre cross-prop diff = 0.0286;  p_perm = 9.999e-05
